# Vision-Driven Computer Use

`06_BrowserAgent_Computer_Use_Applied.ipynb` teaches the shape of a computer-use loop, but
is explicit that it is a simulation: its `screenshot_text()` returns a *text description* of
the page. The agent never sees anything.

This notebook closes that. The agent receives an actual **PNG image** of a UI, and has to
work out from pixels where things are and what to click. Nothing is described to it in
words.

That change is not cosmetic. When the observation is text, someone has already done the
hard part — parsing the interface into named elements. When it is pixels, the model has to
do that itself, and everything that makes computer use unreliable shows up.

## Learning objectives

1. Build an act–observe loop whose observation is an image, not a description.
2. Send a screenshot to a vision model and parse a structured action back.
3. Watch raw-coordinate clicking fail completely, then **measure** two mitigations — a
   coordinate ruler drawn on the screenshot, and a stronger model — instead of assuming
   either works.
4. Tell three different failure classes apart from the click log alone.
5. Use it as a **frontend testing** agent, and report *errored* separately from *failed*.

## Where this fits

- `06_BrowserAgent_Computer_Use_Applied.ipynb` — the same loop with text observations.
  Read it first; this is the pixel version.
- `07_Hosted_vs_Client_Side_Tools.ipynb` — OpenAI's `computer_use_preview` is the hosted
  version of everything below, with the trade-offs that notebook describes.

## Dependencies, deliberately none new

The UI is rendered with **Pillow**, which is already installed, and the screenshots are real
PNGs. No Playwright, no Selenium, no headless Chromium download — none of which are
dependencies of this repo.

What that costs: the GUI is drawn rather than browsed. What it keeps: real pixels, real
vision, real coordinates, and a loop that behaves the way a browser-driven one does. The
final section explains exactly what changes when you swap the renderer for a real browser.

## Prerequisites

`OPENAI_API_KEY` in the project-root `.env`. Every number quoted in this notebook came from
actually running it; a full pass costs roughly 15–20 cents.

In [ ]:
# ============ SETUP ============
import base64
import io
import json

from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont

load_dotenv()

# Vision calls on gpt-4o-mini are enormous in token terms (section 5 measures it), so a
# burst of them will hit a per-minute token limit. Let the SDK back off rather than crash
# the loop half way through a demo.
client = OpenAI(max_retries=5)

MODEL = "gpt-4o"            # the default everywhere below; section 5 shows why
WEAK_MODEL = "gpt-4o-mini"  # the contrast: cheaper per token, useless at this task

# The goal needs 4 successful clicks (+, +, coupon, place) plus a `done`, so a cap of 6
# would decide the outcome rather than bound it. 10 leaves room to recover from misses
# while still stopping a confused agent.
MAX_STEPS = 10


def _font(size: int):
    for path in ("/System/Library/Fonts/Supplemental/Arial.ttf",
                 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                 "C:/Windows/Fonts/arial.ttf"):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()

## 1. A UI made of pixels

`Screen` draws a small checkout page and tracks its own state. The important detail is what
it does **not** expose: there is no `get_elements()`, no accessibility tree, no DOM. The only
way out is `screenshot()`, which returns a PNG.

Clicks arrive as `(x, y)`. The screen decides what, if anything, was hit — exactly as a real
GUI does.

In [ ]:
# ============ THE UI ============
class Screen:
    """A drawn checkout page. Observable only as pixels; clickable only by coordinate."""

    W, H = 520, 360
    GRID = 40          # ruler spacing, in logical units

    def __init__(self, coupon_button_broken: bool = False):
        self.qty = 1
        self.coupon_applied = False
        self.placed = False
        self.coupon_button_broken = coupon_button_broken   # the bug, for section 5
        self.click_log: list[tuple[int, int, str]] = []
        # name -> (x1, y1, x2, y2). The agent never sees this.
        self.boxes = {
            "qty_plus":   (250, 96, 286, 130),
            "qty_minus":  (196, 96, 232, 130),
            "coupon":     (40, 190, 210, 228),
            "place":      (40, 262, 260, 306),
        }

    def _hit(self, x: int, y: int) -> str | None:
        for name, (x1, y1, x2, y2) in self.boxes.items():
            if x1 <= x <= x2 and y1 <= y <= y2:
                return name
        return None

    def click(self, x: int, y: int) -> str:
        """Coordinates are always LOGICAL (0..W, 0..H), whatever the screenshot was scaled to."""
        target = self._hit(x, y)
        self.click_log.append((x, y, target or "MISS"))
        if target == "qty_plus":
            self.qty += 1
        elif target == "qty_minus":
            self.qty = max(1, self.qty - 1)
        elif target == "coupon":
            if not self.coupon_button_broken:
                self.coupon_applied = True
        elif target == "place":
            self.placed = True
        return target or "MISS"

    def _draw(self) -> Image.Image:
        img = Image.new("RGB", (self.W, self.H), "#f4f4f6")
        d = ImageDraw.Draw(img)
        big, mid, small = _font(20), _font(16), _font(13)

        d.text((40, 30), "Checkout", fill="#111", font=big)
        d.text((40, 70), "Wireless Mouse", fill="#333", font=mid)

        # quantity stepper
        d.rectangle(self.boxes["qty_minus"], outline="#555", width=2, fill="white")
        d.text((208, 103), "-", fill="#111", font=mid)
        d.text((240, 103), str(self.qty), fill="#111", font=mid)
        d.rectangle(self.boxes["qty_plus"], outline="#555", width=2, fill="white")
        d.text((262, 103), "+", fill="#111", font=mid)

        # coupon
        fill = "#cdebd6" if self.coupon_applied else "white"
        d.rectangle(self.boxes["coupon"], outline="#555", width=2, fill=fill)
        label = "Coupon applied" if self.coupon_applied else "Apply coupon"
        d.text((54, 202), label, fill="#111", font=mid)

        # place order
        d.rectangle(self.boxes["place"], outline="#1a5", width=3,
                    fill="#1a5" if not self.placed else "#888")
        d.text((60, 277), "Order placed" if self.placed else "Place order",
               fill="white", font=mid)

        d.text((40, 326), f"total: ${12.50 * self.qty:.2f}", fill="#333", font=small)
        return img

    def render(self, grid: bool = False, scale: int = 1) -> Image.Image:
        """`grid` overlays a labelled coordinate ruler; `scale` enlarges the whole image.

        The ruler is labelled in LOGICAL units, so a coordinate read off it is directly
        clickable no matter what `scale` is. Note what the ruler does NOT do: it never
        names or outlines a control. It supplies a reference frame, not an answer key.
        """
        img = self._draw()
        if grid:
            d = ImageDraw.Draw(img)
            f = _font(11)
            for x in range(0, self.W, self.GRID):
                d.line([(x, 0), (x, self.H)], fill="#c9c9d4", width=1)
                d.text((x + 2, 1), str(x), fill="#d0021b", font=f)
            for y in range(self.GRID, self.H, self.GRID):
                d.line([(0, y), (self.W, y)], fill="#c9c9d4", width=1)
                d.text((2, y + 1), str(y), fill="#d0021b", font=f)
        if scale != 1:
            img = img.resize((self.W * scale, self.H * scale), Image.LANCZOS)
        return img

    # kept so `screen.screenshot()` still reads naturally in the plain loop
    screenshot = render


def to_data_url(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


demo = Screen()
print(f"  plain:  {demo.render().size[0]}x{demo.render().size[1]} PNG")
_ruled = demo.render(grid=True, scale=2)
print(f"  ruled:  {_ruled.size[0]}x{_ruled.size[1]} PNG")
demo.render()

## 2. The agent sees pixels

One call: the screenshot goes in as an image, a JSON action comes back. The prompt gives the
model the image dimensions and the action vocabulary — and nothing about what is on screen.

Note the action space is deliberately tiny: `click(x, y)` or `done`. Every extra verb is
another thing the model can get wrong.

In [ ]:
# ============ ONE VISION STEP ============
_CONTRACT = (
    "Reply ONLY with JSON, no prose, no code fences:\n"
    '  {"action": "click", "x": <int>, "y": <int>, "why": "<short>"}\n'
    '  {"action": "done", "why": "<short>"}\n'
    "Click the CENTRE of the control you intend to press."
)

SYSTEM = (
    "You operate a GUI by looking at screenshots. You are given a PNG of the current screen "
    f"({Screen.W}x{Screen.H} pixels, origin top-left). Decide the single next action.\n"
    + _CONTRACT
)

SYSTEM_RULER = (
    "You operate a GUI by looking at screenshots. A coordinate ruler is drawn over the "
    "screenshot: red numbers along the top edge are x, red numbers down the left edge are "
    f"y, and gridlines fall every {Screen.GRID} units.\n"
    f"Read the target's position off those rulers and report it in RULER units "
    f"(x from 0 to {Screen.W}, y from 0 to {Screen.H}) — not in raw image pixels.\n"
    "State which gridlines the control sits between before you commit to a number.\n"
    + _CONTRACT
)


def decide(img: Image.Image, goal: str, history: list[str],
           system: str = SYSTEM, model: str = MODEL) -> dict:
    past = ("\n".join(f"- {h}" for h in history)) or "- (nothing yet)"
    reply = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": [
                {"type": "text",
                 "text": f"Goal: {goal}\n\nActions so far:\n{past}\n\nNext action?"},
                {"type": "image_url", "image_url": {"url": to_data_url(img)}},
            ]},
        ],
    )
    raw = (reply.choices[0].message.content or "").strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"action": "done", "why": f"unparseable reply: {raw[:60]}"}

## 3. The loop

Screenshot → decide → act → screenshot. The agent is told whether its click **hit something
or missed**, never what the thing was called. Missing is information too, and a capable
agent should correct after one.

The first run deliberately uses the **weak** model on a **raw** screenshot — the most
demanding version of the task, and the one everybody reaches for first because the model is
advertised as the cheap one.

In [ ]:
# ============ ACT-OBSERVE LOOP ============
GOAL = "Set the quantity to 3, apply the coupon, then place the order."


def run(goal: str, screen: Screen, max_steps: int = MAX_STEPS, verbose: bool = True,
        grid: bool = False, scale: int = 1, model: str = MODEL):
    system = SYSTEM_RULER if grid else SYSTEM
    history: list[str] = []
    for step in range(1, max_steps + 1):
        action = decide(screen.render(grid=grid, scale=scale), goal, history, system, model)
        if action.get("action") == "done":
            if verbose:
                print(f"  {step}. done — {action.get('why','')}")
            break
        x, y = int(action.get("x", -1)), int(action.get("y", -1))
        hit = screen.click(x, y)
        note = f"clicked ({x},{y}) -> {hit}"
        history.append(note)
        if verbose:
            print(f"  {step}. {note}   [{action.get('why','')[:44]}]")
    return screen


def report(screen: Screen, label: str) -> int:
    misses = sum(1 for *_, t in screen.click_log if t == "MISS")
    met = screen.qty == 3 and screen.coupon_applied and screen.placed
    print(f"\n  [{label}] qty={screen.qty}  coupon={screen.coupon_applied}  "
          f"placed={screen.placed}   goal_met={met}")
    print(f"  [{label}] clicks: {len(screen.click_log)} ({misses} missed)")
    return misses


print(f"GOAL: {GOAL}")
print(f"Model: {WEAK_MODEL}   Observation: raw {Screen.W}x{Screen.H} screenshot\n")
weak_plain = run(GOAL, Screen(), model=WEAK_MODEL)
report(weak_plain, "mini + raw")

### Discussion of the output

Watch the `MISS` count. That number is the entire reliability story of coordinate-based
computer use, and it is why this is the least dependable tool-use pattern in the repo.

This configuration does not miss *often*. Across five runs while writing this notebook it
missed **53 clicks out of 53** — it never once hit a control. A representative trace:

```
1. clicked (350,220) -> MISS   [to increase the quantity to 3]
2. clicked (400,220) -> MISS   [To increase the quantity to 3.]
3. clicked (440,220) -> MISS   [To increase the quantity to 3.]
4. clicked (480,220) -> MISS   [to increase the quantity to 3]
5. clicked (480,220) -> MISS   [To increase the quantity to 3.]
```

Three things are visible there, and the third is the important one:

1. **The intent is correct throughout.** It knows it wants the `+` control. Recognising
   *what* is on screen and knowing *where* it is are different abilities, and vision models
   are markedly better at the first.
2. **Only one axis is wrong, and it is always the same one.** Every click sits at
   `y=220`; the stepper is at `y=96–130`. The model picked a row and never questioned it,
   then searched along `x` — the axis that was already closest to right.
3. **Feeding back `MISS` did not help.** This loop is generous: `run()` appends
   `clicked (x,y) -> MISS` to the history and hands it back on the next call. The agent was
   told it missed, ten times, and never revised `y` once. **Error feedback is not error
   correction.**

A miss is not a crash, and here it is not even silent — and it *still* does not get
corrected. In a real harness you get the next screenshot and nothing else, so the same
failure is genuinely invisible: **not an error, a no-op loop.** The step cap is what turns
it into a bounded failure instead of an unbounded bill.

Two things could fix this: make the observation easier to measure against, or use a better
model. The next two sections try them in that order and **measure both**, because the
obvious guess about which one matters turns out to be wrong.

## 4. Mitigation 1 — give it something to measure against

Nothing about the agent changes. The screen, the model, the action vocabulary, the loop —
all identical. Two things change about the **observation**:

- The screenshot is rendered at **2×**, so each control covers four times the image area.
- A **coordinate ruler** is drawn over it: gridlines every 40 units, labelled along the top
  and left edges.

Be clear about what the ruler is and is not. It does not name a control, outline one, or
number the clickable regions — the agent still has to find the `+` button itself. It
supplies a *reference frame*: instead of estimating "about 350 across, maybe 220 down" from
nothing, the model can read that the button sits between the `240` and `280` gridlines.

This is a stripped-down **Set-of-Mark prompting**, the standard mitigation for weak visual
grounding. Because the ruler is labelled in logical units, a coordinate read off it is
directly clickable regardless of the 2× scaling — the `click()` API never changes.

`max_steps` is cut to 5 here for a reason measured in section 5: a ruled screenshot costs
`gpt-4o-mini` almost 37,000 prompt tokens, so ten of them will blow through a 200k
tokens-per-minute limit.

In [ ]:
# ============ MITIGATION 1: RULED SCREENSHOT, SAME WEAK MODEL ============
print(f"Model: {WEAK_MODEL}   Observation: {Screen.W * 2}x{Screen.H * 2} with a "
      f"coordinate grid\n")
weak_ruled = run(GOAL, Screen(), max_steps=5, grid=True, scale=2, model=WEAK_MODEL)
report(weak_ruled, "mini + ruled")

# what the model was actually looking at
Screen().render(grid=True, scale=2)

### It helps, and it is nowhere near enough

Over four ruled runs on `gpt-4o-mini`: **2 hits in 27 clicks**, against 0 in 53 raw. Real
movement, and still a total failure — the quantity never reached 3, the coupon was never
applied, the order was never placed, in any run.

What changed is *which axis is wrong*, and that is the interesting part:

| | raw screenshot | ruled screenshot |
|---|---|---|
| coupon button is at | x 40–210, y 190–228 | x 40–210, y 190–228 |
| agent clicked | (350, 220), (480, 220) … | (300, **200**), (440, **200**) … |
| verdict | wrong row **and** wrong column | **right row**, wrong column |

The ruler fixed `y`. Ruled clicks land at `y=120` for the stepper row and `y=200` for the
coupon row — both correct. Then `x` goes to 300–440 for a button that ends at 210.

**The mitigation did not remove the error, it moved it.** That is worth internalising
before you ship one: a prompt-level fix that improves a metric can be relocating the
failure rather than removing it, and only the per-click log tells you which. Had we watched
the miss *count* alone we would have recorded a modest improvement and missed that the
character of the failure changed completely.

One run also ended after three clicks with `done — "quantity set and coupon applied"`. None
of that had happened. A model weak enough to mislocate controls is also weak enough to
**claim completion it did not achieve**, which is exactly why section 6 asserts on screen
state rather than trusting the agent's own report.

## 5. Mitigation 2 — use a model that can do it

The other variable. Same loop, same ruler, same everything — `gpt-4o` instead of
`gpt-4o-mini`. We run it both ways to separate the model's contribution from the ruler's.

In [ ]:
# ============ MITIGATION 2: STRONGER MODEL ============
print(f"Model: {MODEL}   Observation: raw {Screen.W}x{Screen.H}\n")
strong_plain = run(GOAL, Screen())
report(strong_plain, "4o + raw")

print(f"\n{'=' * 60}")
print(f"Model: {MODEL}   Observation: ruled 2x\n")
strong_ruled = run(GOAL, Screen(), grid=True, scale=2)
report(strong_ruled, "4o + ruled")

In [ ]:
# ============ WHAT A SCREENSHOT ACTUALLY COSTS ============
# usage.prompt_tokens is ground truth; vision pricing is not intuitive.
def image_tokens(img: Image.Image, model: str) -> int:
    r = client.chat.completions.create(
        model=model, max_tokens=1,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "hi"},
            {"type": "image_url", "image_url": {"url": to_data_url(img)}}]}])
    return r.usage.prompt_tokens


PRICE_PER_MTOK = {"gpt-4o": 2.50, "gpt-4o-mini": 0.15}   # USD / 1M input tokens
s = Screen()

print(f"  {'model':<14}{'screenshot':<16}{'prompt tokens':>14}{'$ / call':>12}")
for model in (WEAK_MODEL, MODEL):
    for label, img in (("raw 520x360", s.render()),
                       ("ruled 1040x720", s.render(grid=True, scale=2))):
        tok = image_tokens(img, model)
        print(f"  {model:<14}{label:<16}{tok:>14,}{tok * PRICE_PER_MTOK[model] / 1e6:>12.4f}")

### The result that reorders the whole notebook

Every run made while writing this notebook, nothing excluded:

| Model | Observation | Runs | Clicks | Missed | Miss rate | Goal met |
|---|---|---|---|---|---|---|
| `gpt-4o-mini` | raw | 5 | 53 | **53** | 100% | **0** |
| `gpt-4o-mini` | ruled 2× | 4 | 27 | 25 | 93% | **0** |
| `gpt-4o` | raw | 3 | 26 | 15 | 58% | 2 |
| `gpt-4o` | ruled 2× | 3 | 22 | 9 | **41%** | 2 |

The best ruled `gpt-4o` run finished the entire goal in **five clicks with zero misses**.
The best `gpt-4o-mini` run never completed anything at all.

**Capability dominates the prompt trick.** Swapping the model takes the miss rate from 100%
to 58% and turns a task that never completes into one that usually does. The ruler helps on
top of that — 58% to 41% — but it cannot manufacture grounding the model does not have. If
you are getting coordinates that are wildly wrong, reach for the model before you reach for
a cleverer prompt.

Treat these as indicative, not as a benchmark: single-digit run counts on one synthetic
screen. The 100%-versus-41% gap is far too wide to be noise, but the gap between the two
`gpt-4o` rows is not something three runs can pin down.

And the cost cell above removes the usual objection. Measured, not estimated:

| | `gpt-4o-mini` | `gpt-4o` |
|---|---|---|
| raw 520×360 | 14,175 tok → **$0.0021** | 433 tok → **$0.0011** |
| ruled 1040×720 | 36,843 tok → **$0.0055** | 1,113 tok → **$0.0028** |

`gpt-4o` is **half the price per screenshot** *and* it works. The mini model bills images at
a far larger token-equivalent per tile, which swamps its cheaper per-token rate. For vision
work the "cheap" model is the expensive one — and it is a rate-limit problem too, since
eight ruled `gpt-4o-mini` calls exhaust a 200k tokens-per-minute bucket.

#### A third kind of failure

Look at the first ruled `gpt-4o` click. Both trials opened with a dead-centre hit on
`qty_minus` — *"Increase quantity to 2"* — when they wanted `qty_plus`.

That is not a grounding failure. The coordinate was perfect. It clicked the wrong control on
purpose, having read `-` as the thing that increments. Three distinct failures now, which
the click log alone tells apart:

| Symptom in the log | Failure | What fixes it |
|---|---|---|
| `MISS`, same row repeatedly | cannot locate | ruler, bigger render, better model |
| `MISS`, right row wrong column | partially located | more marks, or selectors |
| **hit**, but the wrong control | cannot reason about the UI | prompt, labels, better model |

No amount of coordinate help fixes the third row, because nothing about the coordinate was
wrong.

## 6. As a frontend testing agent

This is OpenAI's own example use case for computer use, and it is a better fit than
general automation, because a test has something the open-ended case lacks: **a
verifiable end state.**

Below, the coupon button is wired to do nothing — a plausible frontend regression. The
agent is not told. We give it the same goal and then assert on the result, using the
configuration section 5 established actually works.

In [ ]:
# ============ THE AGENT MEETS A BROKEN BUTTON ============
broken = Screen(coupon_button_broken=True)
print("GOAL (against a build where the coupon button is broken)\n")
run(GOAL, broken, grid=True, scale=2)

print("\n  --- assertions ---")
checks = {
    "quantity is 3":   broken.qty == 3,
    "coupon applied":  broken.coupon_applied,
    "order placed":    broken.placed,
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

# Three outcomes, not two. A failing assertion alone cannot tell you which one you are in.
coupon_clicks = [c for c in broken.click_log if c[2] == "coupon"]
print()
if broken.coupon_applied:
    print("  diagnosis: coupon applied -> the control works.")
elif coupon_clicks:
    print(f"  diagnosis: the agent hit the coupon control {len(coupon_clicks)}x and state "
          f"never changed -> the control is broken, not the agent.")
else:
    print(f"  diagnosis: the agent never reached the coupon control in "
          f"{len(broken.click_log)} clicks -> agent failure, not a UI verdict.")
    print("             This run says NOTHING about whether the button works. "
          "Treat it as an errored test, not a failing one.")

### Why that diagnosis matters

A failing assertion alone is nearly useless: *"coupon not applied"* could mean the button is
broken, or that the agent never found it. Those need different people to fix them.

The click log separates them, and the code above prints all three verdicts rather than only
the interesting one:

| Click log | Verdict | Whose problem |
|---|---|---|
| Hit the control, state changed | **pass** | nobody |
| Hit the control, state unchanged | **fail** — the UI is broken | frontend |
| Never hit the control | **errored** — the UI is unverified | whoever owns the agent |

The third row is the one teams get wrong. It is not a failing test; it is a test that did
not run. Reporting it as a failure sends someone hunting a frontend bug that may not exist —
and reporting it as a pass is worse.

This is not hypothetical. Running section 4's weak configuration against this same broken
build produced three `FAIL`s having never once touched the coupon button. Under a two-state
reporter that is indistinguishable from a genuine regression, and somebody spends an
afternoon on it.

## 7. What changes with a real browser

Everything above is a genuine vision loop — real PNGs, real pixel coordinates, real
misclicks. What a real browser would change:

| | Here | Playwright / Selenium |
|---|---|---|
| Rendering | Pillow draws it | A real engine, real fonts, real layout |
| What can break | Only what I coded | Scroll position, overlays, timing, focus, animation |
| Element access | Coordinates only | Coordinates **and** selectors / accessibility tree |
| Setup | none | `pip install playwright && playwright install chromium` (~300 MB) |

The loop shape does not change, which is the point of learning it here. What changes is the
*failure* surface: real pages move under you, and a screenshot can be stale by the time the
click lands. The ruler survives the move — but on a dense real page, gridlines over content
hurt legibility, which is why production Set-of-Mark systems label *detected elements*
rather than drawing a grid over everything.

**And the practical conclusion:** if a selector or an accessibility tree is available, use
it. Coordinate clicking is the fallback for when nothing else is exposed, not the default.
Production computer-use systems reach for pixels last, not first — hence the hosted
`computer_use_preview` tool in `07_Hosted_vs_Client_Side_Tools.ipynb`, which exists precisely
because doing this well is harder than it looks.

## Key takeaways

1. **Text observations hide the hard part.** Once the screen is pixels, the agent must
   localise controls itself, and that is where computer use actually fails.
2. **Recognising and locating are different skills.** Expect an accurate description of a
   control paired with coordinates tens of pixels off — usually wrong on one axis only.
3. **Telling the agent it missed is not enough.** This loop feeds `-> MISS` straight back,
   and the weak model still held the wrong row for 53 consecutive clicks. Error feedback is
   not error correction.
4. **The signature failure is a no-op**, not an exception, so a step cap is the only bound
   on the loop. Size it above what a perfect run needs, or the cap decides the outcome
   instead of bounding it.
5. **Measure a mitigation; do not assume it.** The coordinate ruler improved the miss count
   and *moved the error from `y` to `x`* rather than removing it. Only the per-click log
   showed that; the summary count looked like plain progress.
6. **Capability beats prompt engineering here, and costs less.** `gpt-4o` finished the goal
   in five clicks with zero misses; `gpt-4o-mini` never finished it at all — while costing
   roughly twice as much per screenshot, because image tokens do not scale with a model's
   headline price. Measure `usage.prompt_tokens` before believing a vision budget.
7. **A precise click on the wrong control is a different bug**, and no coordinate fix
   touches it. Read the click log, not the miss count.
8. **Report three test outcomes, not two**: passed, failed, and *never reached the control*.
   The third is an errored test, and calling it a failure costs somebody an afternoon.
9. **Prefer selectors to coordinates.** Pixels are the fallback for interfaces that expose
   nothing else.

### Next

- `07_Hosted_vs_Client_Side_Tools.ipynb` — `computer_use_preview` runs this loop on the
  provider's side, with the control trade-offs set out there.